In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [25]:
df = pd.read_csv("Retail_Cleaned.csv")

print("Shape:", df.shape)
print(df.head())
print(df.columns.tolist())

Shape: (400916, 8)
   Invoice StockCode                          Description  Quantity  \
0   489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1   489434    79323P                   PINK CHERRY LIGHTS        12   
2   489434    79323W                  WHITE CHERRY LIGHTS        12   
3   489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4   489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [26]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

df = df.dropna(subset=["InvoiceDate"])

In [27]:
df["TotalAmount"] = (
    df["Quantity"] * df["Price"]
)

print(df[["Quantity", "Price", "TotalAmount"]].head())

   Quantity  Price  TotalAmount
0        12   6.95         83.4
1        12   6.75         81.0
2        12   6.75         81.0
3        48   2.10        100.8
4        24   1.25         30.0


In [28]:
df["Date"] = df["InvoiceDate"].dt.date

daily_product_demand = (
    df[df["Quantity"] > 0]
    .groupby(["StockCode", "Date"])["Quantity"]
    .sum()
    .reset_index()
)

daily_product_demand.head()

,StockCode,Date,Quantity
0,10002,2009-12-01,12
1,10002,2009-12-03,6
2,10002,2009-12-04,73
3,10002,2009-12-06,49
4,10002,2009-12-07,2


In [32]:
product_demand = (
    daily_product_demand
    .groupby("StockCode")
    .agg(
        TotalQuantity=("Quantity", "sum"),
        AverageDailyDemand=("Quantity", "mean"),
        DemandStd=("Quantity", "std"),
        ActiveDays=("Date", "nunique")
    )
    .reset_index()
)

product_demand["DemandStd"] = (
    product_demand["DemandStd"].fillna(0)
)

product_demand.head(10)

,StockCode,TotalQuantity,AverageDailyDemand,DemandStd,ActiveDays
0,10002,7796,45.858824,104.716880,170
1,10080,12,2.400000,2.190890,5
2,10109,4,4.000000,0.000000,1
3,10120,471,14.272727,14.543626,33
4,10123C,624,15.600000,34.373365,40
5,10123G,2246,204.181818,444.898150,11
6,10124A,46,3.285714,2.127786,14
7,10124G,20,3.333333,1.632993,6
8,10125,788,14.867925,12.148184,53
9,10133,967,13.246575,18.763722,73


In [33]:
LEAD_TIME_DAYS = 7
SERVICE_FACTOR = 1.65

product_demand["SafetyStock"] = (
    SERVICE_FACTOR
    * product_demand["DemandStd"]
    * np.sqrt(LEAD_TIME_DAYS)
)

product_demand.head(10)

,StockCode,TotalQuantity,AverageDailyDemand,DemandStd,ActiveDays,SafetyStock
0,10002,7796,45.858824,104.716880,170,457.140456
1,10080,12,2.400000,2.190890,5,9.564309
2,10109,4,4.000000,0.000000,1,0.000000
3,10120,471,14.272727,14.543626,33,63.490047
4,10123C,624,15.600000,34.373365,40,150.056569
5,10123G,2246,204.181818,444.898150,11,1942.198274
6,10124A,46,3.285714,2.127786,14,9.288827
7,10124G,20,3.333333,1.632993,6,7.128815
8,10125,788,14.867925,12.148184,53,53.032772
9,10133,967,13.246575,18.763722,73,81.912832


In [34]:
product_demand["ReorderPoint"] = (
    product_demand["AverageDailyDemand"]
    * LEAD_TIME_DAYS
    + product_demand["SafetyStock"]
)

product_demand[
    [
        "StockCode",
        "AverageDailyDemand",
        "SafetyStock",
        "ReorderPoint"
    ]
].head(10)

,StockCode,AverageDailyDemand,SafetyStock,ReorderPoint
0,10002,45.858824,457.140456,778.152221
1,10080,2.400000,9.564309,26.364309
2,10109,4.000000,0.000000,28.000000
3,10120,14.272727,63.490047,163.399138
4,10123C,15.600000,150.056569,259.256569
5,10123G,204.181818,1942.198274,3371.471001
6,10124A,3.285714,9.288827,32.288827
7,10124G,3.333333,7.128815,30.462148
8,10125,14.867925,53.032772,157.108243
9,10133,13.246575,81.912832,174.638860


In [35]:
product_demand["CurrentStock"] = (
    product_demand["AverageDailyDemand"] * 3
)

In [36]:
product_demand["RecommendedReorderQty"] = (
    product_demand["ReorderPoint"]
    - product_demand["CurrentStock"]
).clip(lower=0)

In [37]:
product_demand["Recommendation"] = np.where(
    product_demand["CurrentStock"]
    < product_demand["ReorderPoint"],
    "REORDER",
    "NO REORDER"
)

In [38]:
inventory_recommendations = product_demand[
    [
        "StockCode",
        "TotalQuantity",
        "AverageDailyDemand",
        "DemandStd",
        "SafetyStock",
        "ReorderPoint",
        "CurrentStock",
        "RecommendedReorderQty",
        "Recommendation"
    ]
].copy()

inventory_recommendations.head(20)

,StockCode,TotalQuantity,AverageDailyDemand,DemandStd,SafetyStock,ReorderPoint,CurrentStock,RecommendedReorderQty,Recommendation
0,10002,7796,45.858824,104.716880,457.140456,778.152221,137.576471,640.575751,REORDER
1,10080,12,2.400000,2.190890,9.564309,26.364309,7.200000,19.164309,REORDER
2,10109,4,4.000000,0.000000,0.000000,28.000000,12.000000,16.000000,REORDER
3,10120,471,14.272727,14.543626,63.490047,163.399138,42.818182,120.580956,REORDER
4,10123C,624,15.600000,34.373365,150.056569,259.256569,46.800000,212.456569,REORDER
5,10123G,2246,204.181818,444.898150,1942.198274,3371.471001,612.545455,2758.925547,REORDER
6,10124A,46,3.285714,2.127786,9.288827,32.288827,9.857143,22.431684,REORDER
7,10124G,20,3.333333,1.632993,7.128815,30.462148,10.000000,20.462148,REORDER
8,10125,788,14.867925,12.148184,53.032772,157.108243,44.603774,112.504470,REORDER
9,10133,967,13.246575,18.763722,81.912832,174.638860,39.739726,134.899134,REORDER


In [ ]:
priority_products = (
    inventory_recommendations[
        inventory_recommendations["Recommendation"] == "REORDER"
    ]
    .sort_values(
        "RecommendedReorderQty",
        ascending=False
    )
    .head(20)
)

priority_products

,StockCode,TotalQuantity,AverageDailyDemand,DemandStd,SafetyStock,ReorderPoint,CurrentStock,RecommendedReorderQty,Recommendation
40,16044,6192,3096.000000,4310.522938,18817.543330,40489.543330,9288.000000,31201.543330,REORDER
3602,85220,12236,2447.200000,4055.870215,17705.859499,34836.259499,7341.600000,27494.659499,REORDER
2311,37410,25685,1426.944444,4642.215172,20265.542348,30254.153459,4280.833333,25973.320126,REORDER
505,21092,12806,711.444444,2937.155111,12822.120275,17802.231386,2134.333333,15667.898053,REORDER
2303,37351,5391,898.500000,2187.639618,9550.118141,15839.618141,2695.500000,13144.118141,REORDER
511,21099,13284,458.068966,2404.529995,10496.950837,13703.433595,1374.206897,12329.226699,REORDER
504,21091,13668,414.181818,2252.763989,9834.417908,12733.690636,1242.545455,11491.145181,REORDER
436,20993,10945,456.041667,1896.693702,8279.996749,11472.288416,1368.125000,10104.163416,REORDER
3601,85218,1992,996.000000,1374.615583,6000.870117,12972.870117,2988.000000,9984.870117,REORDER
501,21088,14927,497.566667,1803.122726,7871.513621,11354.480288,1492.700000,9861.780288,REORDER


In [39]:
inventory_recommendations.to_csv(
    "Inventory_Recommendations.csv",
    index=False
)

print("Inventory recommendations saved successfully.")

Inventory recommendations saved successfully.


In [40]:
import os

print(
    os.path.exists("Inventory_Recommendations.csv")
)

True
